In [4]:
!pip install cvxpy
!pip install mosek
!pip install ecos


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\nsukh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\nsukh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\nsukh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import cvxpy as cp
import mosek

In [ ]:
A = np.array([[-1, 2],
              [-1, 3]])
A1 = np.array([[-2, -1],
               [1, -4]])
B = np.eye(2)

n = 2

P = cp.Variable((n, n), symmetric=True)
Q = cp.Variable((n, n), symmetric=True)
Y = cp.Variable((n, n))  # Y = K @ P

# Строгая положительная определённость и отрицательность для стабильности
constraints = [P >> 1e-6, Q >> 1e-6]

A_tilde_P = A @ P + B @ Y
M11 = A_tilde_P + A_tilde_P.T + Q
M12 = P @ A1

LMI = cp.bmat([[M11, M12],
               [M12.T, -Q]])

constraints += [LMI << -1e-6 * np.eye(4)]

problem = cp.Problem(cp.Minimize(0), constraints)

# Основной вариант: ECOS (самый надёжный для LMI)
try:
    problem.solve(solver=cp.ECOS, verbose=True)
    print("Использован солвер: ECOS")
except:
    # Fallback на SCS с агрессивными параметрами для точности
    print("ECOS недоступен, пробуем улучшенный SCS...")
    problem.solve(solver=cp.SCS,
                  eps=1e-10,
                  max_iters=200000,
                  scale=0.1,
                  acceleration_lookback=0,
                  verbose=True)

print("\nСтатус:", problem.status)

if problem.status in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
    P_val = P.value
    Q_val = Q.value
    Y_val = Y.value
    
    K = np.linalg.solve(P_val, Y_val.T).T
    
    print("\nНайденный регулятор K:")
    print(np.round(K, 6))
    
    print("\nМатрица Q:")
    print(np.round(Q_val, 6))
    
    print("\nСобственные значения (A + K):")
    print(np.round(np.linalg.eigvals(A + K), 6))
else:
    print("Решение не найдено")

(CVXPY) Dec 29 03:23:52 AM: Your problem has 12 variables, 24 constraints, and 0 parameters.
(CVXPY) Dec 29 03:23:52 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Dec 29 03:23:52 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Dec 29 03:23:52 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Dec 29 03:23:52 AM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Dec 29 03:23:52 AM: Your problem has 12 variables, 24 constraints, and 0 parameters.
(CVXPY) Dec 29 03:23:52 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Dec 29 03:23:52 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Dec 29 03:23:52 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Dec 29 03:23:52 AM: Your p

                                     CVXPY                                     
                                     v1.7.5                                    
ECOS недоступен, пробуем улучшенный SCS...
                                     CVXPY                                     
                                     v1.7.5                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.10 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University,